# Mini-Batch Hartigan — runtime figures

Regenerates the five runtime figures of the paper from the shipped data and writes them next to
this notebook. Run it from this directory (`minibatch_hartigan_tests/results/`), top to bottom.

| Paper | Figure | Written to |
|---|---|---|
| Section 5.2.1, `fig:runtime-to-convergence` | runtime to convergence | `runtime/runtime_to_convergence_f64.png` |
| Section 5.2.1, `fig:runtime-single-iteration` | runtime per iteration / epoch | `runtime/runtime_per_iteration_f64.png` |
| Appendix, `fig:iteration-counts` | iterations / epochs to convergence | `runtime/iterations_to_convergence_f64.png` |
| Appendix, `fig:heatmap-runtime` | JAX speedup over Fortran AS 136 | `runtime/jax_speedup_heatmap_vs_as136.png` |
| Appendix, `fig:random-partition-runtime` | runtime to convergence, random partition | `random_partition/runtime_to_convergence_random_partition.png` |

**Data** (all float64, all on the same 13,500 datasets: 3 $d$ × 15 $N$ × 3 $K$ × 100 datasets, 50
starting partitions each):

* `data/random_centroid/bestof_cpu_full.csv.gz` — CPU: `numba_online_ls`, `numba_mb_cert`, Fortran AS 136 (`fortran_ls`, `fortran_nols`).
* `data/random_centroid/bestof_jaxtime_rc_29208159.csv.gz` — GPU (H200): `jax_mb_cert`.
* `data/random_partition/bestof_rpart_cpu_26508322.csv.gz` and `bestof_jaxtime_rp_29208159.csv.gz` — the same, random-partition initialization (no Fortran).

The clustering-quality tables are produced by `make_quality_tables_tex.py`, not by this notebook.

Each figure is saved with `fig.savefig(..., dpi=fig.dpi, bbox_inches="tight")`, which is exactly how
Jupyter renders a figure inline, so the files are byte-identical to the notebook's inline images.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160, "display.max_columns", 50)

HERE = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
DATA_RC = os.path.join(HERE, "data", "random_centroid")
DATA_RP = os.path.join(HERE, "data", "random_partition")
FIG_RUNTIME = os.path.join(HERE, "runtime")
FIG_RP = os.path.join(HERE, "random_partition")

cpu = pd.read_csv(os.path.join(DATA_RC, "bestof_cpu_full.csv.gz"))
print(f"CPU {cpu.shape}   impls: {sorted(cpu.impl.unique())}")

NS = sorted(cpu.N.unique()); DS = sorted(cpu.d.unique()); KS = sorted(cpu.K.unique())

GRID_PROPS = [0.25, 0.5, 0.75, 0.9]
CMAP = plt.cm.viridis


def _cells(df):
    return df.groupby(["d", "N", "K"])


def draw(ax, s, d, k, **kw):
    try: v = s.xs((d, k), level=("d", "K")).sort_index()
    except KeyError: return
    v = v.dropna()
    if len(v): ax.plot(v.index.values, v.values, **kw)


def fmt_sp(v):
    # Plain decimals across the whole observed range (0.048 .. 305). `.2g` would render
    # anything >= 100 as 3.1e+02, which is unreadable in a cell annotation.
    if v >= 10:  return f"{v:.0f}"      # 45, 305
    if v >= 1:   return f"{v:.1f}"      # 1.7, 3.7
    return f"{v:.2f}"                   # 0.13, 0.05


def save(fig, path):
    """Write the figure exactly as Jupyter's inline backend renders it."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, format="png", dpi=fig.dpi, bbox_inches="tight",
                facecolor=fig.get_facecolor(), edgecolor=fig.get_edgecolor())
    print("saved", os.path.relpath(path, HERE))

## Float64 throughout — CPU implementations vs float64 JAX, Fortran AS 136 baseline

**Everything here is float64.** CPU: Numba online (`numba_online_ls`) and certified Mini-Batch
(`numba_mb_cert`), and Fortran AS 136 (`fortran_ls`). GPU: certified Mini-Batch in JAX on an
H200 (`jax_mb_cert`, renamed `jax_mb_cert_f64` below).

**The JAX rows are the corrected H200 timing run** (job 29208159, `bestof_jaxtime_*` files), in which
no XLA compilation falls inside a timed region (`experiments/tests/gate_jax_compile_in_clock.py`).

Mini-Batch: **`numba_mb_cert` (CPU, solid circles)** and **`jax_mb_cert_f64` (H200, dashed squares)** —
same colour = same proportion, so solid-vs-dashed at one colour is the CPU-vs-GPU read at a matched
batch size. Online CPU baselines: `numba_online_ls` black, `fortran_ls` (AS 136) grey.

In [ ]:
# ---- the H200 float64 table -------------------------------------------------------------
# The CORRECTED timings: job 29208159 (experiments/jobs/run_bestof_jaxtime_h200.job).
_h64 = pd.read_csv(os.path.join(DATA_RC, "bestof_jaxtime_rc_29208159.csv.gz"))
assert (_h64.init_scheme == "random_centroid").all() and (_h64.impl == "jax_mb_cert").all()
_h64 = _h64.drop_duplicates(
    subset=["init_scheme", "impl", "d", "N", "K", "proportion", "dataset_id"])
_h64["impl"] = "jax_mb_cert_f64"

B64 = pd.concat([cpu, _h64], ignore_index=True)     # the CPU table is already float64 throughout
_MB64 = {"numba_mb_cert", "jax_mb_cert_f64"}
B64["is_mb"] = B64.impl.isin(_MB64)
B64["is_pmax"] = B64.is_mb & np.isclose(B64.proportion, (B64.N - B64.K) / B64.N, rtol=1e-8)

# ---- per-regime curves, and paired speedups with Fortran AS 136 as the denominator ----------
_base64 = B64[B64.impl == "fortran_ls"].set_index(["d", "N", "K", "dataset_id"]).conv_mean_s
_base64 = _base64[~_base64.index.duplicated()]


def curve64(impl, col="conv_mean_s", prop=None, how="mean"):
    # Per-regime value, indexed (d,N,K). MEAN over the 100 datasets: best-of-50 means you
    # genuinely pay for the slow inits, so the mean is the honest cost of one answer.
    g = B64[B64.impl == impl]
    g = g[g.proportion < 0] if prop is None else g[np.isclose(g.proportion, prop) & ~g.is_pmax]
    return _cells(g)[col].mean() if how == "mean" else _cells(g)[col].median()


def speedup64(impl, prop=None, col="conv_mean_s"):
    # PAIRED speedup over Fortran AS 136: ratio per dataset_id, then MEAN over datasets.
    g = B64[B64.impl == impl]
    g = g[g.proportion < 0] if prop is None else g[np.isclose(g.proportion, prop) & ~g.is_pmax]
    g = g.set_index(["d", "N", "K", "dataset_id"])
    r = (_base64.reindex(g.index) / g[col]).dropna()
    return r.groupby(level=["d", "N", "K"]).mean()


print(f"B64: {len(B64):,} rows   impls={sorted(B64.impl.unique())}")
print(f"jax_mb_cert_f64 coverage: "
      f"{_h64.groupby(['d','N','K','dataset_id']).ngroups:,} / 13,500 cells")
print("\npaired median speedup over Fortran AS 136, best grid proportion per cell:")
for _im in ["fortran_nols", "numba_online_ls", "numba_mb_cert", "jax_mb_cert_f64"]:
    _g = B64[B64.impl == _im]
    _g = _g[_g.is_grid] if "is_grid" in _g else _g[~_g.is_pmax]
    if (_g.proportion > 0).any():
        _g = _g[np.isclose(_g.proportion.values[:, None], GRID_PROPS).any(axis=1)]
        _g = _g.loc[_g.groupby(["d", "N", "K", "dataset_id"]).conv_mean_s.idxmin()]
    _g = _g.set_index(["d", "N", "K", "dataset_id"])
    _r = (_base64.reindex(_g.index) / _g.conv_mean_s).dropna().groupby(level="d").median()
    print(f"  {_im:18s} " + "  ".join(f"d={d}: {v:7.2f}x" for d, v in _r.items()))

### Runtime to convergence (paper: `fig:runtime-to-convergence`)

**Rows are d, columns are K**, log-log. `sharey="row"` gives each dimensionality its own vertical
scale. Each point is the mean over the 100 datasets in that regime of the per-dataset mean over 50
starting partitions.

In [ ]:
# ONE figure: rows = d, columns = K, log-log.
#
# sharey="row" is deliberate: convergence time spans orders of magnitude between d=100 and
# d=10000, so one shared y axis across all nine panels would compress every curve into a line.
# Sharing within a row keeps the comparison that matters -- implementations at fixed d -- on a
# common scale.
fig, axes = plt.subplots(len(DS), len(KS), figsize=(5.2 * len(KS), 3.9 * len(DS)),
                         sharex=True, sharey="row")
axes = np.asarray(axes).reshape(len(DS), len(KS))

for ri, d in enumerate(DS):
    for ci, k in enumerate(KS):
        ax = axes[ri, ci]
        first = (ri == 0 and ci == 0)          # legend entries once for the whole figure
        for pi, pp in enumerate(GRID_PROPS):
            col = CMAP(pi / (len(GRID_PROPS) - 1))
            draw(ax, curve64("numba_mb_cert", prop=pp), d, k, color=col, marker="o", ms=3,
                 ls="-", lw=1.8, label=f"numba mb p={pp:g}" if first else None)
            draw(ax, curve64("jax_mb_cert_f64", prop=pp), d, k, color=col, marker="s", ms=3,
                 ls="--", lw=1.8, label=f"jax mb p={pp:g}" if first else None)
        draw(ax, curve64("numba_online_ls"), d, k, color="black", marker="o", ms=4, lw=2.6,
             label="numba online (CPU)" if first else None)
        draw(ax, curve64("fortran_ls"), d, k, color="0.45", marker="^", ms=4, lw=1.8,
             ls="-.", label="Fortran AS 136 (CPU)" if first else None)
        ax.set_xscale("log"); ax.set_yscale("log"); ax.grid(alpha=.3, which="both")
        if ri == 0:
            ax.set_title(f"K = {k}")
        if ri == len(DS) - 1:
            ax.set_xlabel("N")
        if ci == 0:
            ax.set_ylabel(f"d = {d}\nconvergence time (s)")

h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="center left", bbox_to_anchor=(0.99, 0.5), fontsize=9)
fig.suptitle("Runtime to convergence — mini-batch Hartigan against online CPU baselines",
             y=1.005)
plt.tight_layout()
save(fig, os.path.join(FIG_RUNTIME, "runtime_to_convergence_f64.png"))
plt.show(); plt.close(fig)

### JAX speedup heatmap over Fortran AS 136 (paper: `fig:heatmap-runtime`)

Laid out like the runtime figure: **rows of panels are d, columns
are K**; within a panel, rows are N and columns are the batch fraction $p$. Only JAX mini-batch
(`jax_mb_cert_f64`) is shown, against Fortran AS 136 (`fortran_ls`).

Each cell is a **paired mean speedup** (`speedup64`, loader cell above). For each of the 100
datasets in that $(d, N, K)$ regime, AS 136's time to convergence is divided by JAX's time on the
same dataset — each time already the mean over that dataset's 50 initialisations — and the cell
reports the mean of those 100 ratios: the average speedup on one dataset. Because it averages
per-dataset ratios, it is not the same number as dividing the runtime figure's two mean curves.

Log colour scale centred at 1×, so **1× = white, red = faster, blue = slower**; colorbar and cell
annotations both in raw ×. Each row of panels has its own colour scale and colorbar, because
speedups at $d = 10{,}000$ sit far above those at $d = 100$ and one shared scale would wash out the
$d = 100$ row.

Blank cells are grid fractions with $\lceil pN \rceil \ge N - K$: that batch would reach the cap
$N - K$ set by the leave-one-per-cluster constraint, so that fraction is not run there (the JAX
timing run is grid-only and never runs $p_{\max} = (N-K)/N$).

In [ ]:
cols64 = [(im, lab, pp) for im, lab in [("jax_mb_cert_f64", "jax")]
          for pp in GRID_PROPS]
SP64 = {(im, pp): speedup64(im, pp) for im, _, pp in cols64}

# ONE figure, rows = d, columns = K. Each row keeps its own colour scale and colorbar --
# speedups at d=10000 sit far above those at d=100, and one shared scale would wash out d=100.
fig, axes = plt.subplots(len(DS), len(KS), figsize=(5.4 * len(KS), 5.4 * len(DS)),
                         squeeze=False, layout="constrained")
for ri, d in enumerate(DS):
    mats = []
    for k in KS:
        M = np.full((len(NS), len(cols64)), np.nan)
        for ni, n in enumerate(NS):
            for ci, (im, _, pp) in enumerate(cols64):
                M[ni, ci] = SP64[(im, pp)].get((d, n, k), np.nan)
        mats.append(M)
    fin = np.array(mats)[np.isfinite(mats) & (np.array(mats) > 0)]
    R = max(fin.max(), 1.0 / fin.min()) if fin.size else 10.0
    norm = LogNorm(vmin=1.0 / R, vmax=R)
    for ki, (k, M) in enumerate(zip(KS, mats)):
        a = axes[ri][ki]
        hm = a.imshow(M, aspect="auto", cmap="RdBu_r", norm=norm)
        a.set_xticks(range(len(cols64)))
        a.set_xticklabels([f"{pp:g}" for _, _, pp in cols64], fontsize=8)
        a.set_yticks(range(len(NS))); a.set_yticklabels(NS, fontsize=8)
        if ri == 0: a.set_title(f"K = {k}")
        if ri == len(DS) - 1: a.set_xlabel("batch fraction $p$")
        if ki == 0: a.set_ylabel(f"d = {d}\nN")
        for ni in range(len(NS)):
            for ci in range(len(cols64)):
                v = M[ni, ci]
                if np.isfinite(v):
                    a.text(ci, ni, fmt_sp(v), ha="center", va="center", fontsize=7,
                           color="white" if (v > 4 or v < 0.25) else "black")
    fig.colorbar(hm, ax=axes[ri], label="speedup over Fortran AS 136 (x)", fraction=.02)
fig.suptitle("JAX speedup over Fortran AS 136")
save(fig, os.path.join(FIG_RUNTIME, "jax_speedup_heatmap_vs_as136.png"))
plt.show(); plt.close(fig)

### Runtime per iteration and iterations to convergence (paper: `fig:runtime-single-iteration`, `fig:iteration-counts`)

One **single figure** per metric: rows are d, columns are K.

**The two units.** A *mini-batch iteration* reassigns the sampled batch of ⌈p·n⌉ points against
fixed start-of-iteration centroids, then recomputes all K centroids. An *online epoch* is one
sequential sweep of all n points, each reassignment followed immediately by a rank-1 centroid
update. These are not the same unit, and the axis labels say so.

**Why they are nonetheless comparable here — transitively.** A full online sweep and a full batch
sweep consider the same n points, so those are directly comparable. A full batch sweep and a
mini-batch iteration differ only in how many points are touched — same hardware, same kernel
structure — and on this hardware that barely matters: at d=10,000, N=10,000, K=16 the
per-iteration cost rises only from **0.939 ms at p=0.25 to 1.340 ms at p=0.9**, i.e. **1.43× the
time for 3.6× the points**. Additional arithmetic is close to free on this hardware. And p=0.9 already samples
the large majority of the data, so it is very nearly a full sweep at very nearly the cost of a
quarter sweep.

That sub-linear scaling is itself part of the result, and it is why this figure tracks the
runtime-to-convergence figure so closely: on the GPU, convergence *length* explains most of
convergence *time*.

`numba_mb_cert` (CPU, solid, circles) against `jax_mb_cert` (H200, dashed, squares) at each grid
proportion in viridis, with `numba_online_ls` in black and Fortran AS 136 in grey. Per-iteration
runtime is log-scaled with its own vertical scale per row, since it spans orders of magnitude
across d; the count is linear on one shared scale so it stays comparable across the whole grid.

**How `singleit_mean_s` is measured for the JAX rows.** Inside the certified run itself: the clock
starts once initialization is done and covers the mini-batch iterations and the convergence
checks they trigger, divided by the number of iterations — the same definition `numba_mb_cert`
uses, so the two families are measured identically.


In [ ]:
# One figure per metric: rows = d, columns = K. viridis by proportion, log for runtime and
# linear for the count.
#
# UNITS. The mini-batch series are per mini-batch ITERATION (one batch of ceil(p*n) reassignments
# plus a full centroid recompute); the online series are per EPOCH (one sequential sweep of all n
# points). Different units, named as such in every title and axis label -- see the markdown above
# for why they are still comparable on this hardware.
#
# sharey differs by metric on purpose. Per-iteration runtime spans orders of magnitude between
# d=100 and d=10000, so it gets "row" (each dimensionality on its own scale). Counts sit in the
# same tens-to-hundreds range everywhere, so they get one shared scale.
_SERIES_74 = [
    ("numba_mb_cert",   "numba mb",  dict(marker="o", ms=3, ls="-",  lw=1.8)),
    ("jax_mb_cert_f64", "jax mb",    dict(marker="s", ms=3, ls="--", lw=1.8)),
]
_BASE_74 = [
    ("numba_online_ls", "numba online (CPU)",    dict(color="black", marker="o", ms=4, lw=2.6)),
    ("fortran_ls",      "Fortran AS 136 (CPU)",  dict(color="0.45",  marker="^", ms=4, lw=1.8,
                                                     ls="-.")),
]

for col, lab, title, logy, shy, fname in [
        ("singleit_mean_s", "runtime per iteration / epoch (s)",
         "Runtime per mini-batch iteration and per online epoch", True, "row",
         "runtime_per_iteration_f64.png"),
        ("n_iters_mean", "iterations / epochs to convergence",
         "Mini-batch iterations and online epochs to convergence", False, True,
         "iterations_to_convergence_f64.png")]:
    fig, axes = plt.subplots(len(DS), len(KS), figsize=(5.2 * len(KS), 3.9 * len(DS)),
                             sharex=True, sharey=shy)
    axes = np.asarray(axes).reshape(len(DS), len(KS))
    for ri, d in enumerate(DS):
        for ci, k in enumerate(KS):
            ax = axes[ri, ci]
            first = (ri == 0 and ci == 0)          # legend entries once for the whole figure
            for impl, tag, st in _SERIES_74:
                for pi, pp in enumerate(GRID_PROPS):
                    c = CMAP(pi / (len(GRID_PROPS) - 1))
                    draw(ax, curve64(impl, col, pp), d, k, color=c,
                         label=f"{tag} p={pp:g}" if first else None, **st)
            for impl, tag, st in _BASE_74:
                draw(ax, curve64(impl, col), d, k, label=tag if first else None, **st)
            ax.set_xscale("log")
            if logy:
                ax.set_yscale("log")
            ax.grid(alpha=.3, which="both")
            if ri == 0:
                ax.set_title(f"K = {k}")
            if ri == len(DS) - 1:
                ax.set_xlabel("N")
            if ci == 0:
                ax.set_ylabel(f"d = {d}\n{lab}")
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="center left", bbox_to_anchor=(0.99, 0.5), fontsize=9)
    fig.suptitle(title, y=1.005)
    plt.tight_layout()
    save(fig, os.path.join(FIG_RUNTIME, fname))
    plt.show(); plt.close(fig)

### Runtime to convergence — random partition initialization (paper: `fig:random-partition-runtime`)

The runtime-to-convergence construction on the random-partition data: **rows are d, columns are K**,
log-log, each point the mean over the 100 datasets in that regime. The y-limits are pinned to the
random-centroid figure's, row by row, so the two figures can be read side by side. Same datasets —
only the initialization differs. The partition enters each algorithm directly.

Two Mini-Batch families, both float64: `numba_mb_cert` (CPU, solid, circles) and `jax_mb_cert`
(H200, dashed, squares), at each grid proportion in viridis. The online CPU baseline is
`numba_online_ls` in black.

**No Fortran AS 136 curve.** AS 136 cannot start from a given partition — its `KMNS` overwrites the
initial assignment before reading it — so it was not run in the random-partition arm.

In [ ]:
# The runtime-to-convergence figure, rebuilt on the random-partition files. The loader is local to
# this cell so it cannot disturb B64. CPU rows: job 26508322; JAX rows: the corrected H200 timing
# run, job 29208159. The dedup below keeps the first copy of each key (there are no duplicates).
_rpf = [os.path.join(DATA_RP, "bestof_rpart_cpu_26508322.csv.gz"),
        os.path.join(DATA_RP, "bestof_jaxtime_rp_29208159.csv.gz")]
BRP = pd.concat([pd.read_csv(p) for p in _rpf], ignore_index=True).drop_duplicates(
    subset=["init_scheme", "impl", "d", "N", "K", "proportion", "dataset_id"])
assert (BRP.init_scheme == "random_partition").all()
BRP.loc[BRP.impl == "jax_mb_cert", "impl"] = "jax_mb_cert_f64"     # same name as above
BRP["is_pmax"] = (BRP.impl.isin({"numba_mb_cert", "jax_mb_cert_f64"})
                  & np.isclose(BRP.proportion, (BRP.N - BRP.K) / BRP.N, rtol=1e-8))
print(f"random partition: {len(BRP):,} rows   cells per impl: "
      f"{ {i: g.groupby(['d','N','K','dataset_id']).ngroups for i, g in BRP.groupby('impl')} }")


def curve_rp(impl, col="conv_mean_s", prop=None):
    # curve64 over BRP: per-regime MEAN over the 100 datasets, indexed (d, N, K).
    g = BRP[BRP.impl == impl]
    g = g[g.proportion < 0] if prop is None else g[np.isclose(g.proportion, prop) & ~g.is_pmax]
    return _cells(g)[col].mean()


_RPC = {(im, pp): curve_rp(im, prop=pp)
        for im in ("numba_mb_cert", "jax_mb_cert_f64") for pp in GRID_PROPS}
_RPC["numba_online_ls"] = curve_rp("numba_online_ls")

# The random-centroid figure's row-wise y-limits, recovered by drawing its own curves on a
# throwaway figure. It autoscales with sharey="row", so its limits are a function of its data;
# redrawing the same curves reproduces them exactly, with no numbers hardcoded here.
_f71, _a71 = plt.subplots(len(DS), len(KS), sharex=True, sharey="row")
_a71 = np.asarray(_a71).reshape(len(DS), len(KS))
for ri, d in enumerate(DS):
    for ci, k in enumerate(KS):
        for pp in GRID_PROPS:
            draw(_a71[ri, ci], curve64("numba_mb_cert", prop=pp), d, k)
            draw(_a71[ri, ci], curve64("jax_mb_cert_f64", prop=pp), d, k)
        draw(_a71[ri, ci], curve64("numba_online_ls"), d, k)
        draw(_a71[ri, ci], curve64("fortran_ls"), d, k)
        _a71[ri, ci].set_xscale("log"); _a71[ri, ci].set_yscale("log")
YLIM_71 = {d: _a71[ri, 0].get_ylim() for ri, d in enumerate(DS)}
plt.close(_f71)

fig, axes = plt.subplots(len(DS), len(KS), figsize=(5.2 * len(KS), 3.9 * len(DS)),
                         sharex=True, sharey="row")
axes = np.asarray(axes).reshape(len(DS), len(KS))

for ri, d in enumerate(DS):
    for ci, k in enumerate(KS):
        ax = axes[ri, ci]
        first = (ri == 0 and ci == 0)          # legend entries once for the whole figure
        for pi, pp in enumerate(GRID_PROPS):
            col = CMAP(pi / (len(GRID_PROPS) - 1))
            draw(ax, _RPC[("numba_mb_cert", pp)], d, k, color=col, marker="o", ms=3,
                 ls="-", lw=1.8, label=f"numba mb p={pp:g}" if first else None)
            draw(ax, _RPC[("jax_mb_cert_f64", pp)], d, k, color=col, marker="s", ms=3,
                 ls="--", lw=1.8, label=f"jax mb p={pp:g}" if first else None)
        draw(ax, _RPC["numba_online_ls"], d, k, color="black", marker="o", ms=4, lw=2.6,
             label="numba online (CPU)" if first else None)
        ax.set_xscale("log"); ax.set_yscale("log"); ax.grid(alpha=.3, which="both")
        if ri == 0:
            ax.set_title(f"K = {k}")
        if ri == len(DS) - 1:
            ax.set_xlabel("N")
        if ci == 0:
            ax.set_ylabel(f"d = {d}\nconvergence time (s)")
    axes[ri, 0].set_ylim(YLIM_71[d])           # sharey="row" carries it across the row

h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="center left", bbox_to_anchor=(0.99, 0.5), fontsize=9)
fig.suptitle("Runtime to convergence, random partition initialisation", y=1.005)
plt.tight_layout()
save(fig, os.path.join(FIG_RP, "runtime_to_convergence_random_partition.png"))
plt.show(); plt.close(fig)